# 02 — Baseline: TF-IDF + Logistic Regression

Every ML project needs a simple, fast baseline. The fine-tuned transformer and the LLM only make sense if they **beat** it.

- **TF-IDF** turns each sentence into a vector of word / word-pair weights (rare, informative words weigh more).
- **One logistic regression per category** (`OneVsRestClassifier`) → handles multi-label naturally.
- `class_weight="balanced"` compensates for the ~11% unfair rate seen in the EDA.

Protocol: choose hyperparameters and thresholds on **validation**, evaluate on **test only once** at the end.

In [1]:
import time

import joblib
import numpy as np
import pandas as pd

from clausula.baseline import build_pipeline
from clausula.data import load_splits
from clausula.labels import LABELS
from clausula.metrics import per_label, summary, tune_thresholds

splits = load_splits()
X = {name: df["text"] for name, df in splits.items()}
Y = {name: df[LABELS].values for name, df in splits.items()}

/home/davig/clausula/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tune the regularization strength `C` on validation

In [2]:
results = []
for C in [0.1, 0.3, 1, 3, 10, 30, 100, 300]:
    model = build_pipeline(C).fit(X["train"], Y["train"])
    results.append({"C": C, **summary(Y["validation"], model.predict(X["validation"]))})

results = pd.DataFrame(results).round(3)
best_C = results.loc[results["macro_f1"].idxmax(), "C"]
print("best C:", best_C)
results

best C: 30.0


,C,macro_f1,micro_f1,unfair_f1
0,0.1,0.585,0.562,0.686
1,0.3,0.599,0.586,0.702
2,1.0,0.620,0.612,0.720
3,3.0,0.644,0.642,0.731
4,10.0,0.690,0.691,0.763
5,30.0,0.713,0.711,0.759
6,100.0,0.708,0.706,0.748
7,300.0,0.707,0.698,0.745


## 2. Tune one threshold per category

Logistic regression outputs probabilities; the default cut is 0.5. Rare categories often need a different cut, so we pick the threshold that maximizes each category's F1 **on validation**.

In [3]:
model = build_pipeline(best_C).fit(X["train"], Y["train"])

val_probs = model.predict_proba(X["validation"])
thresholds = tune_thresholds(Y["validation"], val_probs)

print("validation, threshold 0.5:", {k: round(v, 3) for k, v in summary(Y["validation"], val_probs >= 0.5).items()})
print("validation, tuned:        ", {k: round(v, 3) for k, v in summary(Y["validation"], val_probs >= thresholds).items()})
pd.Series(thresholds, index=LABELS, name="threshold").round(2)

validation, threshold 0.5: {'macro_f1': 0.713, 'micro_f1': 0.711, 'unfair_f1': 0.759}
validation, tuned:         {'macro_f1': 0.742, 'micro_f1': 0.741, 'unfair_f1': 0.785}


Limitation of liability    0.85
Unilateral termination     0.65
Unilateral change          0.70
Content removal            0.60
Contract by using          0.75
Choice of law              0.75
Jurisdiction               0.50
Arbitration                0.85
Name: threshold, dtype: float64

## 3. Final evaluation on test (only once)

In [4]:
test_pred = model.predict_proba(X["test"]) >= thresholds
test_summary = summary(Y["test"], test_pred)
print({k: round(v, 3) for k, v in test_summary.items()})
per_label(Y["test"], test_pred).round(2)

{'macro_f1': 0.765, 'micro_f1': 0.741, 'unfair_f1': 0.818}


,precision,recall,f1,support
Limitation of liability,0.83,0.63,0.72,38
Unilateral termination,0.72,0.68,0.70,38
Unilateral change,0.81,0.58,0.68,38
Content removal,0.55,0.85,0.67,13
Contract by using,0.90,0.78,0.84,23
Choice of law,0.86,0.92,0.89,13
Jurisdiction,0.86,0.75,0.80,16
Arbitration,1.00,0.71,0.83,7


## 4. Latency

In [5]:
sample = X["test"].tolist()
start = time.perf_counter()
model.predict_proba(sample)
ms_per_sentence = (time.perf_counter() - start) / len(sample) * 1000
print(f"{ms_per_sentence:.3f} ms per sentence on CPU ({len(sample)} sentences)")

0.037 ms per sentence on CPU (1607 sentences)


## 5. What did the model learn?

The largest coefficients show the words/bigrams that push a sentence into each category. This checks the EDA hypothesis that *Jurisdiction* may rely on place names.

In [6]:
vocab = np.array(model.named_steps["tfidf"].get_feature_names_out())
estimators = model.named_steps["clf"].estimators_

pd.DataFrame({
    "top features": {label: ", ".join(vocab[np.argsort(est.coef_[0])[::-1][:10]]) for label, est in zip(LABELS, estimators)}
})

,top features
Limitation of liability,"liability, liable, be liable, for any, any, li..."
Unilateral termination,"terminate, suspend, account, or, notice, termi..."
Unilateral change,"change, modify, may change, we may, may modify..."
Content removal,"remove, content, and or, remove any, permanent..."
Contract by using,"by, terms, using, continued use, continued, by..."
Choice of law,"laws, be governed, governed, governed by, laws..."
Jurisdiction,"courts, jurisdiction, the courts, have jurisdi..."
Arbitration,"arbitration, binding, you and, disputes, bindi..."


## 6. Errors: what does it miss and what does it flag by mistake?

In [7]:
pd.set_option("display.max_colwidth", 200)
test = splits["test"]
true_unfair = Y["test"].any(axis=1)
pred_unfair = test_pred.any(axis=1)

print("Missed unfair clauses (false negatives):", (true_unfair & ~pred_unfair).sum())
display(test.loc[true_unfair & ~pred_unfair, ["text"]].head(5))
print("Fair sentences flagged (false positives):", (~true_unfair & pred_unfair).sum())
display(test.loc[~true_unfair & pred_unfair, ["text"]].head(5))

Missed unfair clauses (false negatives): 42


,text
221,the licenses granted by amazon terminate if you do not comply with these conditions of use or any service terms .
336,we reserve the right to revoke the free trial and put your account on hold in the event that we determine that you are not eligible .
369,"netflix regularly makes changes to the service , including the content library ."
449,"while we 're not required to do so , we may access , review , screen , and delete your content at any time and for any reason , including if we think your content violates these terms ."
464,"we encourage you to give the privacy policy a careful look because , by using our services , you agree that your information may be collected and processed by snap group limited in the uk and snap..."


Fair sentences flagged (false positives): 16


,text
43,your subscription continues until cancelled by you or we terminate your access to or use of the services or the subscription in accordance with these terms .
458,"although snap group limited reserves the right to review all content that appears on the services and to remove any content that violates these terms , we do not necessarily review all of it ."
492,"by using the services , you agree that :"
568,the courts in some countries may not apply the laws of england and wales to some disputes related to these terms .
624,we can not and will not be liable for any loss or damage arising from your failure to comply with the above .


## 7. Save the model

In [8]:
joblib.dump({"model": model, "thresholds": thresholds, "labels": LABELS}, "../models/baseline.joblib")

pd.DataFrame([{"model": "TF-IDF + LogReg", **{k: round(v, 3) for k, v in test_summary.items()},
               "ms/sentence": round(ms_per_sentence, 3), "cost": "free (CPU)"}])

,model,macro_f1,micro_f1,unfair_f1,ms/sentence,cost
0,TF-IDF + LogReg,0.765,0.741,0.818,0.037,free (CPU)


## Conclusions

| Model | macro-F1 | micro-F1 | unfair-F1 | ms/sentence | cost |
|---|---|---|---|---|---|
| TF-IDF + LogReg | **0.765** | 0.741 | 0.818 | ~0.04 (CPU) | free |

1. **Weak regularization works best** (`C = 30`; 100 and 300 get worse). The cues are very lexical (*governed by*, *arbitration*, *liable*), so the model benefits from trusting specific words and bigrams.
2. **Per-category thresholds add ~3 macro-F1 points on validation** (0.713 → 0.742). Cheap and effective.
3. **Strengths:** categories with fixed legal vocabulary — *Choice of law* (0.89), *Contract by using* (0.84), *Arbitration* (0.83), *Jurisdiction* (0.80).
   The *Jurisdiction* place-name worry from the EDA is **not confirmed**: its top features are *courts* and *jurisdiction*, not *santa clara*.
4. **Weaknesses:** *Unilateral change* (0.68), *Content removal* (0.67), *Limitation of liability* (0.72), mostly due to **low recall**. The missed clauses express the idea with less typical words: *"the licenses granted by amazon terminate if…"*, *"netflix regularly makes changes to the service"*. This is exactly where a model that understands context (a transformer) should help.
5. **False positives are mostly near-misses:** *"by using the services, you agree that:"* or *"the courts in some countries may not apply the laws of england"* contain the right words but aren't unfair by themselves.
6. **Caveat:** test has only 7–16 examples in some categories, so per-category scores have high variance (1 error in *Arbitration* = −14 recall points).

**Bar to beat:** the fine-tuned transformer must clearly exceed **0.765 macro-F1** to justify being much slower and needing more compute (we'll measure how much).
